# Interactive feature matching

In [ ]:
import sys
sys.path.append('..')

from multiview_stitcher import param_utils
from multiview_stitcher import spatial_image_utils as si_utils
import numpy as np

from src.muvis_align.metrics import calc_sims_metrics
from src.muvis_align.MVSRegistration import MVSRegistration
from src.muvis_align.image.util import get_sim_physical_size, draw_keypoints_matches, get_overlap_images, affine_from_intrinsic_affine

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_path='../data/S000/*.zarr', output_path='../../output/', ui='mpl', debug=True)
sims = reg.init_sims()
norm_sims, _, _ = reg.preprocess(sims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Get overlap between tiles

In [ ]:
%matplotlib inline
overlap1, overlap2, sims_pixel_space = get_overlap_images(norm_sims[0], norm_sims[1], reg.source_transform_key)
overlap1, overlap2 = overlap1.squeeze().compute(), overlap2.squeeze().compute()
print('Overlap size [um]:', overlap1.shape * si_utils.get_spacing_from_sim(sims[0], asarray=True))

draw_keypoints_matches(overlap1, [], overlap2, [])

## Interactive: Set up feature extraction and matching, and show results

### You can modify the parameters and rerun this cell to update results, e.g.:

**transform_type**: affine, euclidean, translation

**name**: sift, orb

In [ ]:
%matplotlib inline
register_params = {
    'transform_type': 'rigid',
    'pairing': 'orthogonal',
    'name': 'sift',
    'normalisation': True,
    'gaussian_sigma': 1,
    'max_keypoints': 5000,
    'inlier_threshold_factor': 0.05,
    'max_trials': 1000,
    'ransac_iterations': 3,
    'n_parallel_pairwise_regs': 1,
}

reg_method, pairwise_reg_func, pairwise_reg_func_kwargs = reg.create_registration_method(sims[0], params=register_params)
results = pairwise_reg_func(overlap1, overlap2)
print('transform:\n', results['affine_matrix'])

## Run registration on all tiles, showing results corresponding to each tile overlap

In [ ]:
affine_phys = affine_from_intrinsic_affine(results['affine_matrix'], sims_pixel_space, reg.source_transform_key)
transforms = {
    (0, 1): affine_phys
}
qualities = {
    (0, 1): np.array(results['quality'])
}
metrics = calc_sims_metrics(norm_sims[:2], transforms, qualities)
summary = metrics['summary']

print(summary)